# WikiPron Dual-Pronunciation Dataset Builder

This notebook reads `data/g2p/wikipron_filtered.jsonl`, filters for words that have **two possible pronunciations**, samples **50 words**, and expands them into a **100-sample dataset** (2 rows per word, one for each pronunciation), strictly validated against the corpus phoneme inventory.

Outputs (exported to `output/dictee/wikipron_dual_pronunciations/`):
- `sentences.txt`: 100 prompt lines in IPA phonemic notation for recording with [Dictée](https://github.com/benjaminbellamy/Dictee).
- `metadata.csv`: Full 100-sample metadata with contrast classification.


In [1]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

RANDOM_SEED = 42

ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
INPUT_PATH = ROOT / "data" / "g2p" / "wikipron_filtered.jsonl"
OUTPUT_DIR = ROOT / "output" / "dictee" / "wikipron_dual_pronunciations"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", ROOT)
print("Input path   :", INPUT_PATH)
print("Output dir   :", OUTPUT_DIR.relative_to(ROOT))


Project root : /home/lorenzmarqueses/Documents/Git/transcription-correction
Input path   : /home/lorenzmarqueses/Documents/Git/transcription-correction/data/g2p/wikipron_filtered.jsonl
Output dir   : output/dictee/wikipron_dual_pronunciations


## 1. Load `wikipron_filtered.jsonl` & Validate Phoneme Inventory


In [2]:
CONS = set("pbkɡtdʔmŋnɾfvszʃʒhwjl")
VOW = set("aeiouə")
STRESS = "'"
TIE_BAR = "\u0361"
INVENTORY = CONS | VOW | {STRESS, TIE_BAR}


def is_inventory_valid(phoneme_str: str) -> bool:
    if not phoneme_str:
        return False
    return (
        set(phoneme_str).issubset(INVENTORY)
        and not phoneme_str.startswith(TIE_BAR)
        and not phoneme_str.endswith(TIE_BAR)
    )


with INPUT_PATH.open("r", encoding="utf-8") as f:
    raw_rows = [json.loads(line) for line in f]

print(f"Total entries in wikipron_filtered.jsonl: {len(raw_rows):,}")
len_dist = Counter(len(r["prons"]) for r in raw_rows)
print("Pronunciation count distribution:", dict(sorted(len_dist.items())))

dual_rows = [
    r for r in raw_rows
    if len(r["prons"]) == 2
    and r["prons"][0] != r["prons"][1]
    and all(is_inventory_valid(p) for p in r["prons"])
]
print(f"Eligible words with exactly 2 distinct, inventory-valid pronunciations: {len(dual_rows):,}")


Total entries in wikipron_filtered.jsonl: 2,638
Pronunciation count distribution: {2: 2367, 3: 169, 4: 97, 5: 2, 6: 2, 7: 1}
Eligible words with exactly 2 distinct, inventory-valid pronunciations: 2,367


## 2. Sample 50 Words & Expand to 100 Samples


In [3]:
def analyze_contrast(p1, p2):
    s1 = p1.replace("'", "").replace("ʔ", "")
    s2 = p2.replace("'", "").replace("ʔ", "")
    stress_diff = ("'" in p1) != ("'" in p2) or (p1 != p2 and s1 == s2 and "'" in p1 and "'" in p2)
    glottal_diff = ("ʔ" in p1) != ("ʔ" in p2) or (p1.count("ʔ") != p2.count("ʔ"))
    if s1 == s2:
        if stress_diff and glottal_diff:
            return "stress_and_glottal_contrast", f"stress & glottal shift: {p1} vs {p2}"
        elif stress_diff:
            return "stress_contrast", f"stress shift: {p1} vs {p2}"
        elif glottal_diff:
            return "glottal_contrast", f"glottal stop difference: {p1} vs {p2}"
    return "segmental_contrast", f"segmental difference: {p1} vs {p2}"

dual_df = pd.DataFrame(dual_rows)
sampled_words = dual_df.sample(n=50, random_state=RANDOM_SEED).reset_index(drop=True)

records = []
for pair_id, row in sampled_words.iterrows():
    word = row["word"]
    p1, p2 = row["prons"]
    ctype, cdet = analyze_contrast(p1, p2)
    records.append({
        "pair_id": pair_id,
        "word": word,
        "variant_id": 0,
        "phoneme": p1,
        "alternate_phoneme": p2,
        "contrast_type": ctype,
        "contrast_detail": cdet,
        "source_file": "wikipron_filtered.jsonl",
    })
    records.append({
        "pair_id": pair_id,
        "word": word,
        "variant_id": 1,
        "phoneme": p2,
        "alternate_phoneme": p1,
        "contrast_type": ctype,
        "contrast_detail": cdet,
        "source_file": "wikipron_filtered.jsonl",
    })

dataset = pd.DataFrame(records)
dataset.insert(0, "sample_id", range(len(dataset)))

print("Dataset shape:", dataset.shape)
print("Unique words :", dataset["word"].nunique())
print("Variants     :", dataset["variant_id"].value_counts().to_dict())
print("Contrasts    :", dataset["contrast_type"].value_counts().to_dict())
dataset.head(6)


Dataset shape: (100, 9)
Unique words : 50
Variants     : {0: 50, 1: 50}
Contrasts    : {'stress_contrast': 72, 'stress_and_glottal_contrast': 22, 'segmental_contrast': 6}


## 3. Validate & Export for Dictée

We validate inventory compliance and export:
1. `sentences.txt`: 100 IPA prompts (one per line) for recording in [Dictée](https://github.com/benjaminbellamy/Dictee).
2. `metadata.csv`: Full dataset metadata mapped 1-to-1 with prompt indices.


In [4]:
assert len(dataset) == 100, f"Expected 100 samples, got {len(dataset)}"
assert dataset["word"].nunique() == 50, f"Expected 50 unique words, got {dataset['word'].nunique()}"
assert (dataset.groupby("pair_id").size() == 2).all()
assert dataset["phoneme"].apply(is_inventory_valid).all()
assert (dataset["phoneme"] != dataset["alternate_phoneme"]).all()

# 1. Save metadata.csv
METADATA_PATH = OUTPUT_DIR / "metadata.csv"
dataset.to_csv(METADATA_PATH, index=False, encoding="utf-8")

# 2. Save sentences.txt (IPA prompts for Dictée)
SENTENCES_PATH = OUTPUT_DIR / "sentences.txt"
with SENTENCES_PATH.open("w", encoding="utf-8") as f:
    for p in dataset["phoneme"]:
        f.write(f"{p}\n")

print("All 100 samples (50 words x 2 pronunciations) passed validation.")
print(f"Saved metadata to : {METADATA_PATH.relative_to(ROOT)}")
print(f"Saved prompts to  : {SENTENCES_PATH.relative_to(ROOT)}")


All 100 samples (50 words x 2 pronunciations) passed validation.
Saved metadata to : output/dictee/wikipron_dual_pronunciations/metadata.csv
Saved prompts to  : output/dictee/wikipron_dual_pronunciations/sentences.txt


## 4. Dictée Recording & Merging into Manifest JSONL

### Recording with Dictée
Launch Dictée pointing to the exported `sentences.txt`:
```bash
flatpak run fr.benjaminbellamy.dictee output/dictee/wikipron_dual_pronunciations/sentences.txt
```
Dictée records 16 kHz mono WAV files into `sentences/` (`audio_00.wav`..`audio_99.wav`) and maintains `trans.txt`.

### Merging into Manifest JSONL
Once audio is recorded, merge the WAV files and metadata into an ASR manifest (`manifest.jsonl`) using `scripts/merge_dictee.py`:
```bash
python scripts/merge_dictee.py \
    --dictee-dir output/dictee/wikipron_dual_pronunciations/sentences \
    --metadata output/dictee/wikipron_dual_pronunciations/metadata.csv \
    --output output/dictee/wikipron_dual_pronunciations/manifest.jsonl
```


In [5]:
import sys
sys.path.insert(0, str(ROOT))
from scripts.merge_dictee import merge_dictee_session

MANIFEST_PATH = OUTPUT_DIR / "manifest.jsonl"
recording_dir = OUTPUT_DIR / "sentences"

wav_files = list(recording_dir.glob("audio_*.wav")) if recording_dir.exists() else []
if wav_files:
    entries = merge_dictee_session(
        dictee_dir=recording_dir,
        metadata_path=OUTPUT_DIR / "metadata.csv",
        output_manifest_path=MANIFEST_PATH,
        project_root=ROOT,
    )
    print(f"Merged {len(entries)} recordings into: {MANIFEST_PATH.relative_to(ROOT)}")
else:
    print(f"Ready for Dictée recording at: {OUTPUT_DIR.relative_to(ROOT)}/sentences.txt")
    print(f"Once audio is recorded in {recording_dir.relative_to(ROOT)}, re-run this cell to generate manifest.jsonl.")


Ready for Dictée recording at: output/dictee/wikipron_dual_pronunciations/sentences.txt
Once audio is recorded in output/dictee/wikipron_dual_pronunciations/sentences, re-run this cell to generate manifest.jsonl.
